# X-Ray AI Training Project
### Database Download, editing and organizing

In [2]:
import os
import shutil
import random
from google.colab import userdata

# =====================================================================
# 1. ENVIRONMENT SETUP & AUTHENTICATION
# =====================================================================
# Retrieve credentials securely from Colab's Secret Manager
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("📥 Downloading dataset via Kaggle API...")
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia > /dev/null
!unzip -q chest-xray-pneumonia.zip -d dataset/
print("✅ Dataset downloaded and extracted.")

# =====================================================================
# 2. DATASET RESTRUCTURING (FIXING ORIGINAL SPLIT)
# =====================================================================
# The original dataset has a flawed validation split (only 16 images).
# We merge train/val and recreate a robust 80/20 split.
base_dir = '/content/dataset/chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
classes = ['NORMAL', 'PNEUMONIA']

print("🔄 Restructuring data for an 80/20 train/validation split...")

for cls in classes:
    train_imgs = os.listdir(os.path.join(train_dir, cls))
    val_imgs = os.listdir(os.path.join(val_dir, cls))
    all_imgs = train_imgs + val_imgs
    random.shuffle(all_imgs)

    split_idx = int(len(all_imgs) * 0.8)
    new_train, new_val = all_imgs[:split_idx], all_imgs[split_idx:]

    # Move files to proper directories
    for img in val_imgs:
        shutil.move(os.path.join(val_dir, cls, img), os.path.join(train_dir, cls, img))
    for img in new_val:
        shutil.move(os.path.join(train_dir, cls, img), os.path.join(val_dir, cls, img))

print("✅ Data splits fixed.")

📥 Downloading dataset via Kaggle API...
100% 2.29G/2.29G [00:28<00:00, 85.1MB/s]
✅ Dataset downloaded and extracted.
🔄 Restructuring data for an 80/20 train/validation split...
✅ Data splits fixed.


### Model Setup and data loaders

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# =====================================================================
# 3. DATALOADERS & AUGMENTATION (RUN ONCE PER SESSION)
# =====================================================================
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10), # Prevent overfitting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 32
train_loader = DataLoader(datasets.ImageFolder(train_dir, transform=train_transforms), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(datasets.ImageFolder(val_dir, transform=val_transforms), batch_size=batch_size, shuffle=False)

print("✅ DataLoaders created successfully. No need to run this cell again for new experiments.")

✅ DataLoaders created successfully. No need to run this cell again for new experiments.


### Train-Loader and Val-Loader

In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Definir Transformaciones (Data Augmentation y Normalización para ResNet)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10), # Pequeña rotación para evitar overfitting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Cargar Datasets desde las carpetas
train_dir = '/content/dataset/chest_xray/train'
val_dir = '/content/dataset/chest_xray/val'

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)

# 3. Crear los DataLoaders (¡Esto es lo que faltaba!)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"✅ DataLoaders creados con éxito.")
print(f"Lotes de entrenamiento: {len(train_loader)} (de {batch_size} imágenes cada uno)")
print(f"Lotes de validación: {len(val_loader)}")

✅ DataLoaders creados con éxito.
Lotes de entrenamiento: 131 (de 32 imágenes cada uno)
Lotes de validación: 33


### Experiment J: DEEP FINE-TUNING + 448px + HEAVY AUGMENTATION

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader

# =====================================================================
# 🎛️ EXPERIMENT J: DEEP FINE-TUNING + 448px + HEAVY AUGMENTATION
# =====================================================================
EXPERIMENT_NAME = "expJ_densenet_finetuning_448"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Iniciando: {EXPERIMENT_NAME} en {device}")

# 1. HEAVY AUGMENTATION Y RESOLUCIÓN 448x448
transform_train_J = transforms.Compose([
    transforms.Resize((448, 448)), # 🌟 ¡El doble de resolución!
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1), # 🌟 Simula máquinas distintas
    transforms.RandomGrayscale(p=0.1), # 🌟 Fuerza a ignorar tintes de color
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val_J = transforms.Compose([
    transforms.Resize((448, 448)), # Validación solo se redimensiona, no se distorsiona
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. NUEVOS DATALOADERS (Ojo al batch_size=16 para no quemar la RAM)
# ⚠️ Asegúrate de que train_dir y val_dir están definidos en tu Colab
ds_train = datasets.ImageFolder(train_dir, transform=transform_train_J)
ds_val = datasets.ImageFolder(val_dir, transform=transform_val_J)

dl_train = DataLoader(ds_train, batch_size=16, shuffle=True)
dl_val = DataLoader(ds_val, batch_size=16, shuffle=False)

# 3. CARGA DE ARQUITECTURA
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

# 4. UNFREEZING SELECTIVO (Magia de MLOps)
# Congelamos todo primero
for param in model.parameters():
    param.requires_grad = False

# Descongelamos solo los últimos bloques densos para que se especialicen en pulmones
bloques_a_entrenar = ['denseblock3', 'transition3', 'denseblock4', 'norm5']
for name, param in model.named_parameters():
    if any(bloque in name for bloque in bloques_a_entrenar):
        param.requires_grad = True

# Reseteamos el clasificador final (esto siempre va con requires_grad=True)
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 2).to(device)
model = model.to(device)

# 5. OPTIMIZADOR CON LEARNING RATE BAJO (Micro-cirugía)
# Le decimos a Adam que solo actualice los parámetros que hemos descongelado
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

# Usamos CrossEntropyLoss neutra porque nuestra arquitectura ya es lo suficientemente sensible
criterion = nn.CrossEntropyLoss()

print("✅ DenseNet121 configurada a 448px.")
print("✅ Bloques 3 y 4 descongelados.")
print("✅ Aumento de datos extremo activado. ¡Listo para el entrenamiento!")

🚀 Iniciando: expJ_densenet_finetuning_448 en cuda
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 62.2MB/s]


✅ DenseNet121 configurada a 448px.
✅ Bloques 3 y 4 descongelados.
✅ Aumento de datos extremo activado. ¡Listo para el entrenamiento!


### Training Loop with checkpoints

In [6]:
import time
import copy
import torch

num_epochs = 10
best_acc = 0.0
best_loss = float('inf')
# Aquí guardaremos el "cerebro" temporalmente cuando mejore
best_model_wts = copy.deepcopy(model.state_dict())

print(f"🏃‍♂️ Iniciando entrenamiento de {num_epochs} épocas con Checkpointing...")
start_time = time.time()

for epoch in range(num_epochs):
    print(f'\nÉpoca {epoch+1}/{num_epochs}')
    print('-' * 20)

    # Cada época tiene su fase de entrenamiento y validación
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()  # Modelo en modo entrenamiento (aplica Dropout y Batchnorm)
            dataloader = dl_train
        else:
            model.eval()   # Modelo en modo evaluación (apaga Dropout)
            dataloader = dl_val

        running_loss = 0.0
        running_corrects = 0
        total_samples = 0

        # Iteramos sobre los batches de datos
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            # Forward (Solo guardamos gradientes si estamos en 'train')
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                # Backward y optimización solo en entrenamiento
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            # Estadísticas
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)

        # Cálculos finales de la época
        epoch_loss = running_loss / total_samples
        epoch_acc = running_corrects.double() / total_samples

        print(f'👉 {phase.capitalize()} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

        # 🌟 LA MAGIA DEL CHECKPOINTING 🌟
        # Si estamos en validación y hemos roto nuestro récord de Loss, guardamos el modelo
        if phase == 'val' and epoch_loss < best_loss:
            print(f"   🏆 ¡NUEVO RÉCORD! Val Loss bajó de {best_loss:.4f} a {epoch_loss:.4f}.")
            print("   💾 Guardando esta versión del modelo como la mejor...")
            best_loss = epoch_loss
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            # Guardamos el archivo físico por si Colab se corta
            torch.save(model.state_dict(), 'expJ_finetuned_best.pth')

time_elapsed = time.time() - start_time
print('\n' + '='*50)
print(f'✅ Entrenamiento completado en {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
print(f'🌟 Mejor Val Loss: {best_loss:.4f} (Acc: {best_acc:.4f})')

# Al finalizar, le devolvemos al modelo los mejores pesos que encontramos
model.load_state_dict(best_model_wts)
print("🧠 El modelo en memoria ahora tiene los pesos de su mejor época.")
print('='*50)

🏃‍♂️ Iniciando entrenamiento de 10 épocas con Checkpointing...

Época 1/10
--------------------
👉 Train | Loss: 0.1034 | Acc: 0.9591
👉 Val | Loss: 0.0548 | Acc: 0.9790
   🏆 ¡NUEVO RÉCORD! Val Loss bajó de inf a 0.0548.
   💾 Guardando esta versión del modelo como la mejor...

Época 2/10
--------------------
👉 Train | Loss: 0.0577 | Acc: 0.9783
👉 Val | Loss: 0.0440 | Acc: 0.9847
   🏆 ¡NUEVO RÉCORD! Val Loss bajó de 0.0548 a 0.0440.
   💾 Guardando esta versión del modelo como la mejor...

Época 3/10
--------------------
👉 Train | Loss: 0.0411 | Acc: 0.9869
👉 Val | Loss: 0.0495 | Acc: 0.9866

Época 4/10
--------------------
👉 Train | Loss: 0.0383 | Acc: 0.9881
👉 Val | Loss: 0.0411 | Acc: 0.9838
   🏆 ¡NUEVO RÉCORD! Val Loss bajó de 0.0440 a 0.0411.
   💾 Guardando esta versión del modelo como la mejor...

Época 5/10
--------------------
👉 Train | Loss: 0.0352 | Acc: 0.9876
👉 Val | Loss: 0.0387 | Acc: 0.9866
   🏆 ¡NUEVO RÉCORD! Val Loss bajó de 0.0411 a 0.0387.
   💾 Guardando esta versión del

### Test

In [8]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import os

print("🚀 Iniciando Evaluación Final: EXPERIMENTO J (Test Set)...")

# 1. HARDWARE Y MODO EVALUACIÓN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval() # FUNDAMENTAL: Apagar comportamientos aleatorios de entrenamiento

# 2. RUTA DEL TEST SET (El "hospital nuevo")
test_dir = '/content/dataset/chest_xray/test'
if not os.path.exists(test_dir):
    print(f"❌ ERROR: No se encuentra {test_dir}. Revisa la ruta.")

# 3. TRANSFORMACIONES DE TEST (🌟 ¡A 448px!)
# En test NO distorsionamos. Solo redimensionamos a alta resolución y normalizamos.
transform_test_J = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

ds_test = datasets.ImageFolder(test_dir, transform=transform_test_J)
# Batch size de 16 para que la GPU T4 no se quede sin memoria con imágenes de 448px
dl_test = DataLoader(ds_test, batch_size=16, shuffle=False)

# 4. BUCLE DE INFERENCIA Pura
print(f"🔍 Evaluando {len(ds_test)} radiografías gigantes. ¡Cruzando los dedos!...")
all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in dl_test:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 5. RESULTADOS Y MÉTRICAS
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("\n" + "=" * 50)
print("🏆 RESULTADOS TEST SET: DENSENET121 FINE-TUNED (448px)")
print("=" * 50)
print(f"Accuracy General: {accuracy:.4f}")
print(f"   🚨 Falsos Negativos (Enfermos no detectados): {fn}")
print(f"   💸 Falsos Positivos (Sanos marcados): {fp}")
print(f"   ✅ Aciertos: {tp} Neumonías, {tn} Normales")
print("-" * 50)
print(classification_report(all_labels, all_preds, target_names=['Normal', 'Pneumonia']))
print("=" * 50)

🚀 Iniciando Evaluación Final: EXPERIMENTO J (Test Set)...
🔍 Evaluando 624 radiografías gigantes. ¡Cruzando los dedos!...

🏆 RESULTADOS TEST SET: DENSENET121 FINE-TUNED (448px)
Accuracy General: 0.8157
   🚨 Falsos Negativos (Enfermos no detectados): 0
   💸 Falsos Positivos (Sanos marcados): 115
   ✅ Aciertos: 390 Neumonías, 119 Normales
--------------------------------------------------
              precision    recall  f1-score   support

      Normal       1.00      0.51      0.67       234
   Pneumonia       0.77      1.00      0.87       390

    accuracy                           0.82       624
   macro avg       0.89      0.75      0.77       624
weighted avg       0.86      0.82      0.80       624



### TEST with TTA

In [9]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report

print("🚀 Iniciando Evaluación Suprema: DENSENET121 FINE-TUNED (448px) + TTA...")

# 1. HARDWARE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# 2. TRANSFORMACIONES TTA DE ALTA RESOLUCIÓN (🌟 ¡Cuidado con los tamaños!)
test_dir = '/content/dataset/chest_xray/test'

transform_orig = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
transform_rot = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.RandomRotation((10, 10)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
transform_zoom = transforms.Compose([
    transforms.Resize((500, 500)), # Escalamos más grande
    transforms.CenterCrop(448),    # Recortamos al tamaño que espera la red (448)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

ds_test_orig = datasets.ImageFolder(test_dir, transform=transform_orig)
ds_test_rot = datasets.ImageFolder(test_dir, transform=transform_rot)
ds_test_zoom = datasets.ImageFolder(test_dir, transform=transform_zoom)

# Batch size de 16 para imágenes tan grandes
dl_test_orig = DataLoader(ds_test_orig, batch_size=16, shuffle=False)
dl_test_rot = DataLoader(ds_test_rot, batch_size=16, shuffle=False)
dl_test_zoom = DataLoader(ds_test_zoom, batch_size=16, shuffle=False)

# 3. EVALUACIÓN TTA (Consenso)
print(f"🔍 Evaluando {len(ds_test_orig)} pacientes en 3 vías distintas. Tardará un poco...")
all_labels = []
tta_predictions = []

with torch.no_grad():
    for (inputs_o, labels), (inputs_r, _), (inputs_z, _) in zip(dl_test_orig, dl_test_rot, dl_test_zoom):
        inputs_o, inputs_r, inputs_z = inputs_o.to(device), inputs_r.to(device), inputs_z.to(device)
        labels = labels.to(device)

        out_o, out_r, out_z = model(inputs_o), model(inputs_r), model(inputs_z)
        _, pred_o = torch.max(out_o, 1)
        _, pred_r = torch.max(out_r, 1)
        _, pred_z = torch.max(out_z, 1)

        # Hard Voting
        sum_votes = pred_o + pred_r + pred_z
        final_preds = (sum_votes >= 2).int()

        tta_predictions.extend(final_preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 4. RESULTADOS FINALES
cm = confusion_matrix(all_labels, tta_predictions)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)

print("\n" + "=" * 50)
print("🏆 RESULTADOS TEST SET: FINE-TUNED (448px) + TTA")
print("=" * 50)
print(f"Accuracy General: {accuracy:.4f}")
print(f"   🚨 Falsos Negativos (Enfermos no detectados): {fn}")
print(f"   💸 Falsos Positivos (Sanos marcados): {fp}")
print(f"   ✅ Aciertos: {tp} Neumonías, {tn} Normales")
print("-" * 50)
print(classification_report(all_labels, tta_predictions, target_names=['Normal', 'Pneumonia']))

🚀 Iniciando Evaluación Suprema: DENSENET121 FINE-TUNED (448px) + TTA...
🔍 Evaluando 624 pacientes en 3 vías distintas. Tardará un poco...

🏆 RESULTADOS TEST SET: FINE-TUNED (448px) + TTA
Accuracy General: 0.8189
   🚨 Falsos Negativos (Enfermos no detectados): 0
   💸 Falsos Positivos (Sanos marcados): 113
   ✅ Aciertos: 390 Neumonías, 121 Normales
--------------------------------------------------
              precision    recall  f1-score   support

      Normal       1.00      0.52      0.68       234
   Pneumonia       0.78      1.00      0.87       390

    accuracy                           0.82       624
   macro avg       0.89      0.76      0.78       624
weighted avg       0.86      0.82      0.80       624



### Load weights

In [1]:
from google.colab import files

print("📤 Sube aquí tu archivo 'expI1_densenet_baseline.pth'...")
uploaded = files.upload()
print("✅ Archivo subido con éxito. Ya puedes ejecutar la celda de TTA.")

📤 Sube aquí tu archivo 'expI1_densenet_baseline.pth'...


Saving expJ_finetuned_best.pth to expJ_finetuned_best.pth
✅ Archivo subido con éxito. Ya puedes ejecutar la celda de TTA.


### Reconstruct model

In [7]:
import torch
import torch.nn as nn
from torchvision import models
import os

print("⚙️ Iniciando sistema de carga del modelo médico...")

# 1. Detectar hardware (En tu futura API de producción, esto suele ser "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Hardware detectado: {device}")

# 2. Construir el esqueleto vacío de la arquitectura base
print("🏗️ Construyendo el esqueleto de DenseNet121...")
model = models.densenet121(weights=None)

# 3. Modificar la cabeza del modelo para nuestro problema binario (Normal/Neumonía)
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 2)

# 4. Localizar y cargar el archivo de pesos (el "cerebro" entrenado)
weights_path = 'expJ_finetuned_best.pth'
if not os.path.exists(weights_path):
    raise FileNotFoundError(f"❌ ERROR CRÍTICO: No se encuentra el archivo de pesos '{weights_path}'.")

print(f"🧠 Inyectando el conocimiento desde: {weights_path}...")
model.load_state_dict(torch.load(weights_path, map_location=device))

# 5. Pasar el modelo al dispositivo y bloquearlo en modo inferencia
model = model.to(device)
model.eval()

print("✅ Modelo cargado con éxito y bloqueado en modo estricto de evaluación (eval). ¡Listo para diagnosticar!")

⚙️ Iniciando sistema de carga del modelo médico...
🖥️ Hardware detectado: cuda
🏗️ Construyendo el esqueleto de DenseNet121...
🧠 Inyectando el conocimiento desde: expJ_finetuned_best.pth...
✅ Modelo cargado con éxito y bloqueado en modo estricto de evaluación (eval). ¡Listo para diagnosticar!


### TEST with TTA (5 Pipes)

In [8]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report

print("🚀 Iniciando EL COMITÉ MÉDICO: TTA de 5 Vías (448px)...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

test_dir = '/content/dataset/chest_xray/test'

# 1. LAS 5 PERSPECTIVAS
t_orig = transforms.Compose([
    transforms.Resize((448, 448)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
t_rot_r = transforms.Compose([
    transforms.Resize((448, 448)), transforms.RandomRotation((15, 15)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
t_rot_l = transforms.Compose([
    transforms.Resize((448, 448)), transforms.RandomRotation((-15, -15)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
t_zoom = transforms.Compose([
    transforms.Resize((512, 512)), transforms.CenterCrop(448), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
# 🌟 Alteramos fuertemente el contraste simulando otra máquina
t_contrast = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. LOS 5 DATALOADERS (Batch pequeño para no explotar la RAM)
ds_o = datasets.ImageFolder(test_dir, transform=t_orig)
ds_rr = datasets.ImageFolder(test_dir, transform=t_rot_r)
ds_rl = datasets.ImageFolder(test_dir, transform=t_rot_l)
ds_z = datasets.ImageFolder(test_dir, transform=t_zoom)
ds_c = datasets.ImageFolder(test_dir, transform=t_contrast)

dl_o = DataLoader(ds_o, batch_size=8, shuffle=False)
dl_rr = DataLoader(ds_rr, batch_size=8, shuffle=False)
dl_rl = DataLoader(ds_rl, batch_size=8, shuffle=False)
dl_z = DataLoader(ds_z, batch_size=8, shuffle=False)
dl_c = DataLoader(ds_c, batch_size=8, shuffle=False)

# 3. LA VOTACIÓN
print(f"🔍 El Comité está revisando {len(ds_o)} casos. Evaluando...")
all_labels = []
tta_preds = []

with torch.no_grad():
    for (b_o, lbls), (b_rr, _), (b_rl, _), (b_z, _), (b_c, _) in zip(dl_o, dl_rr, dl_rl, dl_z, dl_c):
        lbls = lbls.to(device)

        # Extraer predicciones binarias (0 o 1) de cada modelo
        _, p_o = torch.max(model(b_o.to(device)), 1)
        _, p_rr = torch.max(model(b_rr.to(device)), 1)
        _, p_rl = torch.max(model(b_rl.to(device)), 1)
        _, p_z = torch.max(model(b_z.to(device)), 1)
        _, p_c = torch.max(model(b_c.to(device)), 1)

        # SUMA DE VOTOS (Hard Voting)
        sum_votes = p_o + p_rr + p_rl + p_z + p_c

        # 🌟 REGLA: Si 3 o más "médicos" dicen neumonía, es neumonía.
        final_preds = (sum_votes >= 3).int()

        tta_preds.extend(final_preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

# 4. EL VEREDICTO
cm = confusion_matrix(all_labels, tta_preds)
tn, fp, fn, tp = cm.ravel()
acc = (tp + tn) / (tp + tn + fp + fn)

print("\n" + "=" * 50)
print("🏆 RESULTADOS SUPREMOS: TTA 5 VÍAS (Comité Médico)")
print("=" * 50)
print(f"Accuracy General: {acc:.4f}")
print(f"   🚨 Falsos Negativos: {fn}")
print(f"   💸 Falsos Positivos: {fp}")
print(f"   ✅ Aciertos: {tp} Neumonías, {tn} Normales")
print("-" * 50)
print(classification_report(all_labels, tta_preds, target_names=['Normal', 'Pneumonia']))

🚀 Iniciando EL COMITÉ MÉDICO: TTA de 5 Vías (448px)...
🔍 El Comité está revisando 624 casos. Evaluando...

🏆 RESULTADOS SUPREMOS: TTA 5 VÍAS (Comité Médico)
Accuracy General: 0.8157
   🚨 Falsos Negativos: 0
   💸 Falsos Positivos: 115
   ✅ Aciertos: 390 Neumonías, 119 Normales
--------------------------------------------------
              precision    recall  f1-score   support

      Normal       1.00      0.51      0.67       234
   Pneumonia       0.77      1.00      0.87       390

    accuracy                           0.82       624
   macro avg       0.89      0.75      0.77       624
weighted avg       0.86      0.82      0.80       624

